# P0 ? T5 stage 1 paper-faithful (CVE-only, target=`target`)

M?c ti?u: ch?y ??ng protocol paper cho stage 1, kh?ng ??a CAPEC v?o input.

- Input: **CVE description**.
- Output: full execution flow `Explore / Experiment / Exploit` (`target`).
- Split c? ??nh: 44.331 train / 11.089 validation.
- T5-base, max input 512, max target 512, LR `5e-5`, AdamW, beam 4.
- Schedule **50 epoch**; l?n ch?y n?y d?ng ? **epoch 10** ?? ki?m tra h?i t? v? ti?t ki?m GPU quota.
- Checkpoint 5 v? 10 ???c probe tr?n 1.000 d?ng; k?t qu? headline ch?y full 11.089 d?ng.

Kh?ng c? exploit source code trong dataset; ch? description/metadata.


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os

hits = sorted(glob.glob("/kaggle/input/**/flow_train.jsonl", recursive=True))
print("flow_train hits:", hits)
if not hits:
    raise FileNotFoundError("flow_train.jsonl not found under /kaggle/input")
FLOW_DIR = Path(hits[0]).parent
os.environ["FLOW_DIR"] = str(FLOW_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("FLOW_DIR:", FLOW_DIR)
for name in ["flow_train.jsonl", "flow_validation.jsonl", "flow_stats.json"]:
    p = FLOW_DIR / name
    print(name, p.exists(), f"{p.stat().st_size / 1024 / 1024:.1f} MiB" if p.exists() else "")
    if not p.exists():
        raise FileNotFoundError(p)


In [ ]:
!python -m torch.distributed.run --nproc_per_node=2 scripts/train_t5.py \
  --dataset "$FLOW_DIR" \
  --output-dir /kaggle/working/t5_stage1_paper_target \
  --model-name t5-base \
  --target-field target \
  --max-input-length 512 \
  --max-target-length 512 \
  --per-device-batch-size 4 \
  --gradient-accumulation-steps 8 \
  --learning-rate 5e-5 \
  --num-train-epochs 50 \
  --stop-after-epoch 10 \
  --save-steps 3465 \
  --eval-steps 693 \
  --logging-steps 50 \
  --save-total-limit 2 \
  --no-load-best-model-at-end \
  --fp16 \
  --gradient-checkpointing


In [ ]:
from pathlib import Path
import os
base = Path("/kaggle/working/t5_stage1_paper_target")
final = base / "final"
ckpts = sorted(base.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
print("final exists:", final.exists())
print("checkpoints:", [p.name for p in ckpts])
if not final.exists():
    raise FileNotFoundError(f"final model not found under {base}")
os.environ["CKPT"] = str(final)
print("CKPT:", final)


In [ ]:
# Probe selected checkpoints on 1.000 validation rows.
# This is diagnostic only; the headline result below is the full 11.089-row validation.
import json
import subprocess
from pathlib import Path

base = Path("/kaggle/working/t5_stage1_paper_target")
for step in [3465, 6930]:
    ckpt = base / f"checkpoint-{step}"
    out = Path(f"/kaggle/working/t5_stage1_paper_target_eval1000_ckpt{step}")
    if not ckpt.exists():
        print("skip missing", ckpt)
        continue
    cmd = [
        "python", "scripts/evaluate_t5.py",
        "--checkpoint", str(ckpt),
        "--dataset", str(Path(__import__('os').environ['FLOW_DIR'])),
        "--split", "validation",
        "--target-field", "target",
        "--output-dir", str(out),
        "--max-input-length", "512",
        "--max-target-length", "512",
        "--batch-size", "4",
        "--num-beams", "4",
        "--limit", "1000",
        "--fp16",
    ]
    print("RUN", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


In [ ]:
# Paper-style full validation evaluation: all 11.089 rows, sentence-average BLEU + ROUGE.
!python scripts/evaluate_t5.py \
  --checkpoint "$CKPT" \
  --dataset "$FLOW_DIR" \
  --split validation \
  --target-field target \
  --output-dir /kaggle/working/t5_stage1_paper_target_eval_full \
  --max-input-length 512 \
  --max-target-length 512 \
  --batch-size 4 \
  --num-beams 4 \
  --fp16


In [ ]:
import json
from pathlib import Path

base = Path("/kaggle/working")
for label, p in [
    ("ckpt3465 (1000 rows)", base / "t5_stage1_paper_target_eval1000_ckpt3465" / "metrics.json"),
    ("ckpt6930 (1000 rows)", base / "t5_stage1_paper_target_eval1000_ckpt6930" / "metrics.json"),
    ("final (full validation)", base / "t5_stage1_paper_target_eval_full" / "metrics.json"),
]:
    print("\n====", label, "====")
    if p.exists():
        m = json.loads(p.read_text(encoding="utf-8"))
        print(json.dumps(m, indent=2, ensure_ascii=False))
    else:
        print("missing", p)

print("\n--- sample full-eval predictions ---")
pred = base / "t5_stage1_paper_target_eval_full" / "predictions.jsonl"
if pred.exists():
    with pred.open(encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            row = json.loads(line)
            print(f"\n[{i}] {row['cve_id']}")
            print("REF:", row["reference"][:500])
            print("PRED:", row["prediction"][:500])


## ??c k?t qu?

- `bleu` l? BLEU trung b?nh t?ng c?u, thang 0?1, theo Algorithm 2 c?a paper.
- `bleu_corpus` ch? ?? tham kh?o.
- ROUGE c? Precision / Recall / F1 cho ROUGE-1, ROUGE-2, ROUGE-L.
- L?n ch?y n?y l? **10 epoch ??u c?a schedule 50 epoch** (warmup v? LR schedule v?n t?nh cho 50 epoch), kh?ng ph?i k?t qu? cu?i c?ng c?a paper.
